# Bond Pricing Analytics Walkthrough
This notebook intends to demonstrate the Python-based bond pricing analytics project that I have created.

This program:
1. Loads the bond input data (CSV in this case) and normalizes column names
2. Parses and validates bond records, including dates and bond day-count conventions
3. Computes accrued interest, dirty price, YTM, and duration metrics
4. Builds summary stats and price-yield sensitivity data
5. Exports the results to Excel with clean formatting

This notebook will walk you through these workflow steps.

## Imports

In [7]:
import argparse, math, pandas
from dataclasses import dataclass
from datetime import date
from typing import Any, Dict, Iterable, List, Optional, Tuple
from openpyxl import load_workbook
from openpyxl.chart import LineChart, Reference
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

## Step 1: Loading bond data and normalizing column names
The program begins by importing data from a CSV file (sample_bonds.csv) that contains the following:
* bond_id
* bond_type
* face_value
* coupon_rate
* frequency
* maturity_date
* settlement_date
* market_clean_price
* tax_rate
* day_count

In [9]:
def normalize_columns(df: pandas.DataFrame)-> pandas.DataFrame:
    df = df.copy()
    df.columns = [str(col).strip().lower().replace(' ', '_') for col in df.columns]
    rename_map = {
        'id': 'bond_id',
        'bond': 'bond_id',
        'type': 'bond_type',
        'par': 'face',
        'principal': 'face',
        'coupon': 'coupon_rate',
        'clean_price': 'market_clean_price',
        'price': 'market_clean_price',
        'maturity_years': 'years_to_maturity',
        'years': 'years_to_maturity',
        'freq': 'frequency',
        'maturity': 'maturity_date',
        'maturity_dt': 'maturity_date',
        'settlement': 'settlement_date',
        'settlement_dt': 'settlement_date',
        'daycount': 'day_count',
        'day_count_convention': 'day_count',
    }
    return df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

This block is used to standardize the column names so that the program can rely on this consistent set of headers. Mapping all the different labels in messy, real data files is beneficial to the program, which now only has to deal with a same set of official names.

## Step 2: Parsing and validating bond records, including dates and bond day-count conventions
The next step is to take all of the rows of the input table and turn it into a Bond object. Missing and invalid data is also checked. All errors within the rows are collected and reported.

In [11]:
def parse_bonds(df: pandas.DataFrame, default_tax_rate: Optional[float])-> List[Bond]:
    bonds: List[Bond] = []
    errors: List[str] = []
    for idx, row in df.iterrows():
        bond_id = str(row.get('bond_id') or row.get('id') or f'Bond-{idx + 1}')
        bond_type = str(row.get('bond_type') or 'corporate').strip().lower()
        if bond_type in {'corp', 'corporate'}:
            bond_type = 'corporate'
        elif bond_type in {'muni', 'municipal', 'municipality'}:
            bond_type = 'municipal'
        try:
            face_value = to_float(row.get('face')) or 100.0
            coupon_rate = to_float(row.get('coupon_rate'))
            years_to_maturity = to_float(row.get('years_to_maturity'))
            frequency = int(to_float(row.get('frequency')) or 2)
            market_clean_price = to_float(row.get('market_clean_price'))
            tax_rate = to_float(row.get('tax_rate'))
            maturity_date = to_date(row.get('maturity_date'))
            settlement_date = to_date(row.get('settlement_date'))
            day_count = normalize_day_count(row.get('day_count'))
            if coupon_rate is None:
                raise ValueError('coupon_rate is required')
            if years_to_maturity is None:
                if maturity_date is None:
                    raise ValueError('years_to_maturity or maturity_date is required')
                if settlement_date is None:
                    raise ValueError('settlement_date is required when using maturity_date')
                if settlement_date >= maturity_date:
                    raise ValueError('settlement_date must be before maturity_date')
                years_to_maturity = year_fraction(settlement_date, maturity_date, day_count)
            if market_clean_price is None:
                raise ValueError('market_clean_price is required')
            if frequency <= 0:
                raise ValueError('frequency must be positive')
            if tax_rate is None:
                tax_rate = default_tax_rate
            bonds.append(
                Bond(
                    bond_id=bond_id,
                    bond_type=bond_type,
                    face_value=face_value,
                    coupon_rate=coupon_rate,
                    years_to_maturity=years_to_maturity,
                    frequency=frequency,
                    market_clean_price=market_clean_price,
                    tax_rate=tax_rate,
                    maturity_date=maturity_date,
                    settlement_date=settlement_date,
                    day_count=day_count,
                )
            )
        except Exception as exc:
            errors.append(f'{bond_id}: {exc}')
    if errors:
        raise ValueError('Invalid bond rows:\n' + '\n'.join(errors))
    return bonds

Containers are created: bonds holds valid Bond objects, and errors will hold error messages. The program loops through each row, and converts raw values into proper types to make sure numbers and dates are what they're supposed to be (e.g. to_date(...)). It checks to make sure that everything is valid (e.g. coupon_rate must exist), and if tax rate is missing it uses default. If everything is valid, Bond(...) is created, and if anything goes wrong the error is stored. If any rows fail, the program raises a combined error that lists them all.

Then, date math and day-count conventions are handled.

In [12]:
def add_months(base: date, months: int)-> date:
    return (pandas.Timestamp(base) + pandas.DateOffset(months=months)).date()

def day_count_30_360(start: date, end: date)-> int:
    d1 = 30 if start.day == 31 else start.day
    d2 = 30 if (end.day == 31 and d1 == 30) else end.day
    return (end.year - start.year) * 360 + (end.month - start.month) * 30 + (d2 - d1)

def year_fraction(start: date, end: date, convention: str)-> float:
    if end <= start:
        return 0.0
    if convention == '30/360':
        return day_count_30_360(start, end) / 360.0
    if convention == 'ACT/360':
        return (end - start).days / 360.0
    if convention == 'ACT/365':
        return (end - start).days / 365.0
    if convention == 'ACT/ACT':
        total = 0.0
        cursor = start
        while cursor < end:
            next_year_start = date(cursor.year + 1, 1, 1)
            segment_end = min(next_year_start, end)
            days_in_year = 366 if pandas.Timestamp(cursor.year, 12, 31).is_leap_year else 365
            total += (segment_end - cursor).days / days_in_year
            cursor = segment_end
        return total
    return (end - start).days / 365.0

def previous_coupon_date(maturity: date, settlement: date, frequency: int)-> date:
    months = int(round(12 / frequency))
    current = maturity
    while current > settlement:
        current = add_months(current, -months)
    return current


def accrued_interest(face_value: float, coupon_rate: float, frequency: int, settlement: date, maturity: date, convention: str)-> float:
    prev_coupon = previous_coupon_date(maturity, settlement, frequency)
    next_coupon = add_months(prev_coupon, int(round(12 / frequency)))
    accrual = year_fraction(prev_coupon, settlement, convention)
    period = year_fraction(prev_coupon, next_coupon, convention)
    if period <= 0:
        return 0.0
    coupon = face_value * coupon_rate / frequency
    return coupon * (accrual / period)

This code uses all of the day-count conventions in bonds (30/360, ACT/360, ACT/365, ACT/ACT) and converts date ranges into year fractions. It finds coupon boundaries and then sets up accrued interest computation.

## Step 3: Computing accrued interest, dirty price, YTM, and duration metrics
The next step is to convert a single bond's inputs into reportable pricing, yield, and risk metrics.

In [19]:
def compute_bond_metrics(bond: Bond)-> Dict[str, Any]:
    accrued = 0.0
    if bond.maturity_date and bond.settlement_date:
        accrued = accrued_interest(
            bond.face_value,
            bond.coupon_rate,
            bond.frequency,
            bond.settlement_date,
            bond.maturity_date,
            bond.day_count,
        )
    price_dirty = bond.market_clean_price + accrued
    ytm_from_price = get_ytm_from_price(
        price_dirty,
        bond.face_value,
        bond.coupon_rate,
        bond.years_to_maturity,
        bond.frequency,
    )
    ytm_quoted = round(ytm_from_price, 7)
    price_from_yield = price_from_ytm(
        bond.face_value,
        bond.coupon_rate,
        ytm_quoted,
        bond.years_to_maturity,
        bond.frequency,
    )
    ytm = ytm_from_price
    if price_dirty is None or ytm is None:
        raise ValueError(f'Unable to compute price and YTM for bond {bond.bond_id}.')
    macaulay, modified = duration(
        bond.face_value,
        bond.coupon_rate,
        ytm,
        bond.years_to_maturity,
        bond.frequency,
    )
    price_clean = price_dirty - accrued
    coupon_payment = bond.face_value * bond.coupon_rate / bond.frequency
    tax_equivalent_yield = None
    if bond.bond_type == 'municipal' and bond.tax_rate is not None:
        if bond.tax_rate >= 1:
            raise ValueError(f'Invalid tax_rate for bond {bond.bond_id}.')
        tax_equivalent_yield = ytm / (1 - bond.tax_rate)
    return {
        'bond_id': bond.bond_id,
        'bond_type': bond.bond_type,
        'face': bond.face_value,
        'coupon_rate': bond.coupon_rate,
        'years_to_maturity': bond.years_to_maturity,
        'frequency': bond.frequency,
        'clean_price': price_clean,
        'coupon_payment': coupon_payment,
        'dirty_price': price_dirty,
        'accrued_interest': accrued,
        'ytm': ytm,
        'macaulay_duration': macaulay,
        'modified_duration': modified,
        'tax_rate': bond.tax_rate,
        'tax_equivalent_yield': tax_equivalent_yield,
        'price_from_ytm': price_from_yield,
        'ytm_from_price': ytm_from_price,
        'price_diff': None if price_from_yield is None else price_dirty - price_from_yield,
        'maturity_date': bond.maturity_date,
        'settlement_date': bond.settlement_date,
        'day_count': bond.day_count,
    }

This code works by computing accrued interest and adding it to the market clean price. It solves for the yield that makes the bond's present value equal to dirty price. It also computes the duration metrics including Macaulay duration and modified duration. Then, the coupon payment is calculated, and if it's a municipal bond and a tax rate is provided, it converts the yield to a tax-equivalent yield. All metrics are bundled into a row for the output DataFrame.

## Step 4. Building summary stats and price-yield sensitivity data
The program then summarizes the portfolio and generates the yield-shock data which is needed to draw price-sensitivity charts.

In [20]:
def build_summary(df: pandas.DataFrame)-> pandas.DataFrame:
    metrics = [
        ('Total Bonds', len(df)),
        ('Average Dirty Price', df['dirty_price'].mean()),
        ('Average YTM', df['ytm'].mean()),
        ('Average Macaulay Duration', df['macaulay_duration'].mean()),
        ('Average Modified Duration', df['modified_duration'].mean()),
    ]
    return pandas.DataFrame(metrics, columns=['Metric', 'Value'])

def sensitivity_blocks(df: pandas.DataFrame, bps_range: int, bps_step: int)-> List[Dict[str, Any]]:
    blocks: List[Dict[str, Any]] = []
    step = bps_step / 10000
    span = bps_range / 10000
    for _, row in df.iterrows():
        base = row['ytm']
        yields = [base + delta for delta in frange(-span, span, step)]
        yields = [max(-0.95, y) for y in yields]
        prices = [
            price_from_ytm(
                row['face'],
                row['coupon_rate'],
                y,
                row['years_to_maturity'],
                int(row['frequency']),
            )
            for y in yields
        ]
        blocks.append({'bond_id': row['bond_id'], 'yields': yields, 'prices': prices})
    return blocks

def frange(start: float, stop: float, step: float)-> Iterable[float]:
    current = start
    while current <= stop + 1e-12:
        yield current
        current += step

The build_summary(df) function creates a summary table with statistics like average dirty price, average YTM, and so on. The sensitivity_blocks(df, bps_range, bps_step) build the price vs yield curves for each bond, which will later be used to write the sensitivity sheet and charts in Excel.

## Step 5: Exporting the results to Excel with clean formatting
Now, the program exports the results to Excel with proper formatting for each column.

In [17]:
def apply_number_formats(ws, start_row: int, end_row: int)-> None:
    for row in range(start_row, end_row + 1):
        ws.cell(row=row, column=1).number_format = '0.00%'
        ws.cell(row=row, column=2).number_format = '$#,##0.00'

def autosize_worksheet(ws)-> None:
    for column_cells in ws.columns:
        max_length = 0
        column_letter = column_cells[0].column_letter
        for cell in column_cells:
            if cell.value is None:
                continue
            value_length = 10 if isinstance(cell.value, date) else len(str(cell.value))
            if value_length > max_length:
                max_length = value_length
        if max_length:
            ws.column_dimensions[column_letter].width = max_length + 2

def apply_bonds_number_formats(ws)-> None:
    header_map = {ws.cell(row=1, column=col).value: col for col in range(1, ws.max_column + 1)}
    formats = {
        'years_to_maturity': '0.000',
        'coupon_payment': '0.00',
        'clean_price': '0.000',
        'accrued_interest': '0.000',
        'dirty_price': '0.000',
        'ytm': '0.0000%',
        'tax_equivalent_yield': '0.0000%',
        'macaulay_duration': '0.000',
        'modified_duration': '0.000',
        'price_from_ytm': '0.000',
        'ytm_from_price': '0.0000%',
        'price_diff': '0.0000',
    }
    for header, number_format in formats.items():
        col_idx = header_map.get(header)
        if not col_idx:
            continue
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = number_format

def write_excel_report(bonds_df: pandas.DataFrame, summary_df: pandas.DataFrame, blocks: List[Dict[str, Any]], output_path: str)-> None:
    with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
        bonds_df.to_excel(writer, sheet_name='Bonds', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        pandas.DataFrame().to_excel(writer, sheet_name='Sensitivity', index=False)
        pandas.DataFrame().to_excel(writer, sheet_name='Charts', index=False)
    wb = load_workbook(output_path)
    ws_bonds = wb['Bonds']
    ws_sensitivity = wb['Sensitivity']
    ws_charts = wb['Charts']
    apply_bonds_number_formats(ws_bonds)
    header_map = {ws_bonds.cell(row=1, column=col).value: col for col in range(1, ws_bonds.max_column + 1)}
    for date_col in ('maturity_date', 'settlement_date'):
        col_idx = header_map.get(date_col)
        if col_idx:
            ws_bonds.column_dimensions[get_column_letter(col_idx)].width = 14
            for row in range(2, ws_bonds.max_row + 1):
                cell = ws_bonds.cell(row=row, column=col_idx)
                if isinstance(cell.value, date):
                    cell.number_format = 'yyyy-mm-dd'
    chart_row = 1
    for block in blocks:
        start_row = ws_sensitivity.max_row + 2 if ws_sensitivity.max_row > 1 else 1
        ws_sensitivity.cell(row=start_row, column=1, value=f'Bond {block['bond_id']} Price Sensitivity').font = Font(bold=True)
        header_row = start_row + 1
        ws_sensitivity.cell(row=header_row, column=1, value='Yield').font = Font(bold=True)
        ws_sensitivity.cell(row=header_row, column=2, value='Price').font = Font(bold=True)
        data_start = header_row + 1
        for i, (yield_value, price_value) in enumerate(zip(block['yields'], block['prices'])):
            ws_sensitivity.cell(row=data_start + i, column=1, value=yield_value)
            ws_sensitivity.cell(row=data_start + i, column=2, value=price_value)
        data_end = data_start + len(block['yields']) - 1
        apply_number_formats(ws_sensitivity, data_start, data_end)
        chart = LineChart()
        chart.title = f'{block['bond_id']} Price vs Yield'
        chart.y_axis.title = 'Price'
        chart.x_axis.title = 'Yield'
        data_ref = Reference(ws_sensitivity, min_col=2, min_row=header_row, max_row=data_end)
        cats_ref = Reference(ws_sensitivity, min_col=1, min_row=data_start, max_row=data_end)
        chart.add_data(data_ref, titles_from_data=True)
        chart.set_categories(cats_ref)
        chart.height = 7
        chart.width = 15
        ws_charts.add_chart(chart, f'A{chart_row}')
        chart_row += 16
    autosize_worksheet(ws_bonds)
    autosize_worksheet(ws_sensitivity)
    autosize_worksheet(ws_charts)
    autosize_worksheet(wb['Summary'])
    wb.save(output_path)

def load_input(path: str)-> pandas.DataFrame:
    return pandas.read_excel(path) if path.lower().endswith(('.xlsx', '.xls')) else pandas.read_csv(path)

Formatting helpers are included, such as apply_number_formats(ws, start_row, end_row), which formats the sensitivity table (column 1 as percentage and column 2 as currency). autosize_worksheet(ws) automatically adjusts column widths so text and numbers don't show with ####. apply_bonds_number_formats(ws) applies the specific decimal formatting to each column.

The Excel report is written, and the input data is loaded.

## Quick Summary

We loaded the bond inputs, applied day‑count conventions to compute accrued interest, and solved for dirty price‑based YTM and duration metrics. We also generated summary statistics and price‑yield sensitivity data. This output is saved to `bond_report.xlsx`